# Milestone 5 tour: immutable source, external evidence

This tour follows the canonical clamp repair in six small steps. The declaration path stays at source, patch, evaluator, budget, and search vocabulary. Returned lineage entries provide optional inspection after the run.

## 1. Load immutable source

The source tree contains exact UTF-8 text. The tour stages only the declared tracked `candidate/clamp.py` bytes into its temporary evidence directory before loading. This isolates the tour from checkout-generated `__pycache__` without adding ignore rules: production `load_source_tree` still rejects every undeclared file.

In [1]:
from pathlib import Path
from shutil import copyfile
from tempfile import TemporaryDirectory
import sys

import meta_evolve as meta

example = Path.cwd()
if not (example / "candidate").is_dir():
    example = example / "examples" / "05_python_function"
temporary = TemporaryDirectory()
evidence = Path(temporary.name)
staged = evidence / "candidate"
staged.mkdir()
copyfile(example / "candidate" / "clamp.py", staged / "clamp.py")
source = meta.load_source_tree(staged)
print("loaded immutable source:", tuple(source) == ("clamp.py",))


loaded immutable source: True


## 2. Declare the exact-file repair

The patch names one file, commits the parent text it expects, and supplies its replacement. A conflict, invalid path, or unchanged replacement is a typed proposal outcome.

In [2]:
FIXED = """\
def clamp(value, low, high):
    return max(low, min(value, high))
"""

def repair(tree, context):
    return meta.TextPatch(
        path="clamp.py",
        expected=tree["clamp.py"],
        replacement=FIXED,
    )

patch = repair(source, None)
print("repair path:", patch.path)
print("repair matches parent:", patch.expected == source["clamp.py"])
print("repair changes content:", patch.replacement != patch.expected)


repair path: clamp.py
repair matches parent: True
repair changes content: True


## 3. Declare the independent evaluator

The grader remains outside the candidate tree, and the fixed `run_grader.py` owns the exit protocol: exit `0` passes, exit `1` is a rankable task failure, and any other exit is a typed evaluator failure.


In [3]:
evaluator = meta.CommandEvaluator(
    command=meta.ProcessSpec(
        argv=(sys.executable, str(example / "run_grader.py"),
              str(example / "grader.py")),
        environment={"PYTHONDONTWRITEBYTECODE": "1"},
    ),
    environment=meta.EvaluationEnvironment(
        name="python-clamp-tests",
        version="1",
    ),
)
print("evaluator:", type(evaluator).__name__)
print("environment: python-clamp-tests@1")


evaluator: CommandEvaluator
environment: python-clamp-tests@1


## 4. Declare one budget

Two evaluations cover the broken seed and one successor. The same budget is passed to the task and run.

In [4]:
budget = meta.Budget(evaluations=2, trials=1)
task = meta.Task(
    evaluator=evaluator,
    artifact=meta.SourceTree,
    budget=budget,
)
print(f"budget: evaluations={budget.evaluations} trials={budget.trials}")

budget: evaluations=2 trials=1


## 5. Run one local Greedy step

Proposal and evaluation each receive a fresh workspace. Durable storage retains the committed history, while the live local callbacks remain non-portable.

In [5]:
history = evidence / "history"
with meta.Storage.durable(history) as storage:
    experiment = meta.Experiment(
        task=task,
        seed=source,
        proposer=meta.PatchProposer(repair),
        search=meta.Greedy(max_trials=1),
    )
    result = meta.run(experiment, storage=storage, mode="local")
    best = result.best()
    lineage = result.lineage()
print("local run complete:", len(lineage) == 2)

local run complete: True


## 6. Inspect the result and durable boundary

The public lineage query is progressive disclosure: it returns immutable lineage entries for inspection without putting kernel construction types into the declaration story. Content digests and semantic outcomes are deterministic; observed process wall time is intentionally omitted.

In [6]:
digests = [step.artifact.digest for step in lineage]
scores = [step.metrics["score"] for step in lineage]
exits = [
    next(item.data["exit_code"] for item in step.evidence
         if item.kind == "process-result")
    for step in lineage
]
failures = [type(step.failure).__name__ if step.failure else None for step in lineage]
kinds = [[item.kind for item in step.evidence] for step in lineage]
print("source digests:", digests)
print("scores:", scores)
print("process exits:", exits)
print("failures:", failures)
print("evidence kinds:", kinds)
print("repaired best:", best.value["clamp.py"] == FIXED)
print("parent unchanged:", source["clamp.py"] != FIXED)
print("lineage length:", len(lineage))


source digests: ['659bfe4a9f389018a5059f9045d6db37592c017591629d6baf5a7ab52792ca53', '08abe248cf356c0ee0322aa9329598221640460d6cf0202228097d7f4c4c21c0']
scores: [0.0, 1.0]
process exits: [1, 0]
failures: [None, None]
evidence kinds: [['evaluation-environment', 'process-result'], ['evaluation-environment', 'process-result']]
repaired best: True
parent unchanged: True
lineage length: 2


In [7]:
with meta.Storage.durable(history) as reopened:
    durable = meta.Run(result.id, reopened)
    print("reopened best:", durable.best() == best)
    print("reopened lineage:", durable.lineage() == lineage)
    try:
        meta.resume(result.id, storage=reopened, registry=meta.Registry())
    except meta.CapabilityError as error:
        if "not resumable" not in str(error):
            raise
        print("cold resume: CapabilityError (local run is not resumable)")
    else:
        raise RuntimeError("local workspace run unexpectedly resumed")

reopened best: True
reopened lineage: True
cold resume: CapabilityError (local run is not resumable)


## Failure boundary: a broken grader is not a low score

Candidate-caused breakage remains the rankable score `0` seen above. This separate probe removes the grader itself: runner exit `2` becomes `EvaluatorFailure`, with no metric to rank.

In [8]:
broken_evaluator = meta.CommandEvaluator(
    command=meta.ProcessSpec(
        argv=(sys.executable, str(example / "run_grader.py"),
              str(example / "missing-grader.py")),
        environment={"PYTHONDONTWRITEBYTECODE": "1"},
    ),
    environment=meta.EvaluationEnvironment(name="python-clamp-tests", version="1"),
)
fault = broken_evaluator(source)
process = next(item for item in fault.evidence if item.kind == "process-result")
print("grader fault:", type(fault.failure).__name__)
print("grader fault rankable:", bool(fault.metrics))
print("grader fault process exit:", process.data["exit_code"])
print("grader fault evidence kinds:", [item.kind for item in fault.evidence])
temporary.cleanup()
print("temporary history disposed:", True)


grader fault: EvaluatorFailure
grader fault rankable: False
grader fault process exit: 2
grader fault evidence kinds: ['evaluation-environment', 'process-result']
temporary history disposed: True
